# Let's play around with period detection

We will use:
- Discrete Cosine Transforms


**NOTE:** This will be done on mainly the raw data!

In [43]:
from circadian_cell_painting import(
    detect,
    OscillationResults,
    cosinor_fitted_long
)
from circadian_cell_painting.osc_detection import DampedCosinorDetection

from pathlib import Path
import polars as pl
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
import string
from tqdm import tqdm
import numpy as np
from scipy.signal import savgol_filter
import pandas as pd
from natsort import natsorted
import matplotlib.colors as mcolors
from scipy.fftpack import dct
import numpy.typing as npt

raw_data = Path("/home/derfelt/PartchLabFiles/Circadian "
                "Rhythms/Live Study/enVision/enVision_fo"
                "rmatted/2026-05-11._.FORMATTED.tsv")


raw_df = pl.read_csv(raw_data, separator="\t")

platemap = (
    pl.read_excel("/home/derfelt/PartchLabFiles/Circadian "
                  "Rhythms/Live Study/platemaps/2026-05-11"
                  "Echo_transfer.xlsx", engine='openpyxl')
    .select(["384_Well", 
             "MoleculeID", 
             "strain",
             "sample_type"])
    .unique(subset=["384_Well"])
    .with_columns(
        pl.col("MoleculeID").str.strip_chars()
    )
).rename({"384_Well": "Well"})

meta_cols = ["MoleculeID", "strain"]
wells = raw_df['Well'].unique().to_list()

raw_df = raw_df.join(platemap, on='Well', how='left')
raw_df = (
    raw_df.with_columns(
        pl.concat_str(
            [pl.col(c).fill_null('nan').cast(pl.String) for c in meta_cols],
            separator=' | ',
        ).alias('group_label')
    )
)


display(raw_df)

Well,CalcResultI,Signal,Timepoint,MoleculeID,strain,sample_type,group_label
str,i64,i64,f64,str,str,str,str
"""A01""",20,4,0.0,null,null,null,"""nan | nan"""
"""A02""",20,4,0.0,null,null,null,"""nan | nan"""
"""A03""",20,4,0.0,null,null,null,"""nan | nan"""
"""A04""",40,8,0.0,null,null,null,"""nan | nan"""
"""A05""",40,8,0.0,null,null,null,"""nan | nan"""
…,…,…,…,…,…,…,…
"""P20""",40,8,138.5,null,null,null,"""nan | nan"""
"""P21""",20,4,138.5,null,null,null,"""nan | nan"""
"""P22""",20,4,138.5,null,null,null,"""nan | nan"""


In [16]:
def dct_period_detection(values: npt.NDArray[np.float64], 
                         sampling_interval: float) -> tuple[npt.NDArray, npt.NDArray, npt.NDArray]:
    """
    Runs a discrete cosinor 

    Args:
        values (npt.NDArray[np.float64]): Timeseries measurement (not timepoints)
        sampling_interval (float): the amount of reads per hour

    Returns:
        tuple[npt.NDArray, npt.NDArray, npt.NDArray]: returns the periods, frequency, and amplitudes
    """
    transformed = dct(values, norm="ortho")
    
    # grab the amplitude (the abs of the transformed data)
    amplitude = np.abs(transformed)
    
    N = len(values)
    k = np.arange(N)
    
    # sampling frequency
    fs = 1 / sampling_interval
    
    f = fs * k / (2 * N)
    
    # convert to periods
    
    with np.errstate(divide='ignore'):
        periods = np.where(f > 0, 1 / f, np.inf)
    
    # print(periods, f, amplitude)
    
    return periods, f, amplitude
    

In [39]:
def add_period_marks(ax, x_min, x_max, interval=12):
    marks = np.arange(np.ceil(x_min / interval) * interval, x_max + interval, interval)
    for mark in marks:
        ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
    ax.set_xticks(marks)
    ax.set_xticklabels([f'{int(m)}h' for m in marks], fontsize=7)

well_col = "Well"
timepoint_col = "Timepoint"
feature_col = 'CalcResultI'

period_range = (18, 40)

wells = raw_df['Well'].unique().to_list()

with PdfPages("dct_plots.pdf") as pdf:

    for well in tqdm(wells, desc="wells"):
        well_df = (raw_df
                .filter(pl.col(well_col) == well)
                .sort(timepoint_col)
                )
        
        data = well_df[feature_col].to_numpy()
        period, freq, amplitude = dct_period_detection(data, 0.5)
        
        fig, axs = plt.subplots(2, 1, figsize=(14,8), sharex=False, sharey=False)
        
        axs[0].plot(freq, amplitude)
        axs[0].set_title("freq vs amplitude")
        axs[0].set_xlabel("freq")
        axs[0].set_ylabel("amplitude")
        
        mask = (period >= period_range[0]) & (period <= period_range[1])
        axs[1].plot(period[mask], amplitude[mask])
        axs[1].set_xlabel("period")
        axs[1].set_ylabel("amplitude")
        
        # detected period
        peak_idx = np.argmax(amplitude[mask])
        detected_period = period[mask][peak_idx]
        axs[1].axvline(detected_period , color='red', label=f'Detected: {detected_period:.1f}h')
        
        axs[1].set_title(f"period vs amplitude\n"
                         f"period range: ({period_range[0]}, {period_range[1]})\n"
                         f"possible detected period: {detected_period:.1f}")
        
        add_period_marks(axs[1], period_range[0], period_range[1], interval=1)
        
        fig.suptitle(f"Well: {well}")
        
        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)


wells: 100%|██████████| 384/384 [01:10<00:00,  5.43it/s]
